## 1. Importación de las librerías

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from statsmodels.formula.api import ols

from statsmodels.stats.diagnostic import linear_rainbow
from scipy.stats import ttest_1samp
from statsmodels.stats.stattools import durbin_watson
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.stats import shapiro

In [ ]:
datos_inmuebles = pd.read_excel('./data/Venta_Inmuebles.xlsx')

In [ ]:
data = datos_inmuebles.copy()

## 3. Limpieza inicial de datos.



In [ ]:
data["Fecha"] = pd.to_datetime(data["Fecha"], errors="coerce")

# Ordenar por Id y Fecha
data = data.sort_values(["Id", "Fecha"])

# Para cada Id nos quedamos con el registro más reciente
data = data.drop_duplicates(subset="Id", keep="last")

Este código cambia el tipo de dato de la variable `Fecha`, lo cual es absolutamente permitido (ya que cambia el tipo, no el formato). Este cambio se hace para comparar las fechas y quedarnos con los registros más actualizados que compartan un `Id`.


A continuación, verificamos si todavía tenemos más de un registro con el mismo `Id`:

In [ ]:
# ids duplicados
dup_counts = (data['Id'].value_counts()
                        .loc[lambda s: s > 1]
                        .sort_values(ascending=False))
for id_, n in dup_counts.items():
    print(f"Id={id_} → {n} apariciones")

if dup_counts.sum() == 0:
    print("No hay duplicados")

Como vemos, ya cumplimos con la **unicidad** (no tenemos registros duplicados, ni con el mismo `Id`. Ahora si, ya podemos pasar a separar el conjunto en train y test.

 ## 4. Partición de los datos.

Una vez realizadas las tareas previas de limpieza, podemos separar los datos en dos conjuntos: entrenamiento (**train**), que se utilizará para entrenar el modelo de Regresión Lineal, y prueba (**test**), destinado a evaluar su desempeño. En esta etapa también se definen las variables del modelo, donde la variable objetivo o dependiente es el `Precio`, y el resto de las variables se consideran inicialmente como variables predictoras, aclarando que esta selección podrá ajustarse en etapas posteriores del análisis.

In [ ]:
target = "Precio"
X = data.drop(columns=[target])
y = data[target]

En el código anterior realizamos la separación de las variables, donde `X` contiene todas las variables independientes que servirán como insumo para predecir el valor de la variable objetivo. A continuación, verifiquemos que esta separación se haya realizado correctamente.

In [ ]:
X

In [ ]:
y


Una vez confirmada la correcta separación de las variables, estamos listos para dividir el conjunto de datos en entrenamiento (**train**) y prueba (**test**). Esta decisión depende del modelador; en este caso, se utilizará el 80 % de los datos para entrenar el modelo y el 20 % restante para evaluarlo. Esta proporción permite contar con suficiente información para el aprendizaje, al tiempo que se reserva un conjunto de datos no visto para medir el rendimiento en nuevos ejemplos. Para realizar esta división se emplea la función `train_test_split()`, que recibe las variables explicativas `X`, la variable objetivo `y`, la proporción del conjunto de prueba (`test_size=0.2`) y el parámetro `random_state`, que garantiza que la partición sea reproducible y que los resultados puedan compararse entre ejecuciones.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=1)


Adicionalmente, podemos ver el tamaño de cada conjunto:


In [ ]:
X_train.shape, y_train.shape

In [ ]:
X_test.shape, y_test.shape

## 5. Construcción del pipeline

Antes de construir el `Pipeline`, es importante comprender su propósito. Un `Pipeline` permite encadenar de forma ordenada todas las etapas de un modelo, desde la limpieza y transformación de los datos hasta el entrenamiento, garantizando que estos pasos se ejecuten siempre en el mismo orden y de la misma manera. Esto permite aplicar exactamente las mismas transformaciones tanto a los datos de entrenamiento como a los de prueba, favoreciendo la consistencia y la capacidad de generalización del modelo. Para su construcción, el primer paso es definir claramente las etapas del flujo y las variables que participarán en la predicción. 
Como primer paso, vamos a pensar en las columnas que no deseamos incluir en la predicción, aquellas que consideramos que no aportan información significativa para estimar el `Precio`. En este caso serán las variables `Id`, `Fecha` y `CodigoPostal`.

In [ ]:
cols_to_drop = ["Id", "Fecha", "CodigoPostal"]

def drop_columns(df):
    return df.drop(columns=cols_to_drop, errors="ignore")

dropper = FunctionTransformer(drop_columns)


Adicionalmente, es fundamental identificar las columnas numéricas y categóricas, ya que el tipo de variable determina los pasos de transformación que se aplicarán a cada una.


In [ ]:
numeric_features = [
    "Cuartos", "Baños", "AreaHabitable", "AreaLote", "Pisos",
    "Vista", "Condición", "Calificación", "AreaSuperior",
    "AreaBase", "Años", "Renovación", "Latitud", "Longitud",
    "AreaHabitable15", "AreaLote15"
]

categorical_features = [
    "FrenteMar"
]

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler()),
    ]
)

In [ ]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="if_binary")),
])


Para abordar el atributo de calidad **validez**, se definió una transformación sobre la variable `Condición`, que presentaba valores fuera del rango $[0,4]$, como valores negativos y dígitos repetidos. Según la validación con el negocio, estos casos se corrigieron usando el valor absoluto y reduciendo los dígitos repetidos a uno solo (por ejemplo, 44 → 4), mediante una función integrada en el pipeline.


In [ ]:
def limpiar_condicion(df):
    df = df.copy()
    df["Condición"] = df["Condición"].abs()
    df["Condición"] = df["Condición"].astype(str).str[0].astype(int)
    return df

limpieza_condicion = FunctionTransformer(limpiar_condicion)


Para abordar el atributo de **consistencia** en la variable `FrenteMar`, se definió una transformación personalizada que unifica los valores llevándolos a mayúsculas. De este modo, representaciones equivalentes como "no", " No " o "NO" quedan normalizadas en un único formato. Esta función se integró al Pipeline mediante FunctionTransformer, permitiendo su aplicación automática dentro del flujo de procesamiento.


In [ ]:
def normalizar_frentemar(df):
    df = df.copy()
    df["FrenteMar"] = df["FrenteMar"].str.upper()
    return df

normalizar_frentemar_tr = FunctionTransformer(normalizar_frentemar)


En este paso usamos `ColumnTransformer` para aplicar diferentes transformaciones según el tipo de variable. La idea es decirle explícitamente al modelo qué hacer con las columnas numéricas y qué hacer con las categóricas, dentro de un solo bloque de preprocesamiento.


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)


Ahora definimos `pipeline_regresion` como un pipeline completo de extremo a extremo, que encadena todas las transformaciones sobre los datos junto con el modelo de regresión lineal. Se tienen todos los pasos que definimos anteriormente, secuenciadas en el orden que queremos que se ejecute:


In [ ]:
pipeline_regresion = Pipeline(steps=[
    ("dropper", dropper),
    ("limpiar_condicion", limpieza_condicion),
    ("normalizar_frentemar", normalizar_frentemar_tr),
    ("preprocesamiento", preprocessor),
])

En esta etapa podemos visualizar la estructura de nuestro pipeline:

In [ ]:
from sklearn import set_config
set_config(display="diagram")

In [ ]:
pipeline_regresion

Este pipeline se ajusta a X_train y se utiliza para transformar este conjunto de datos. Vamos a utilizar otra variable para no afectar el conjunto de entrenamiento original.

In [ ]:
Xt_train = pipeline_regresion.fit_transform(X_train)

Un paso importante acá es tener en cuenta que, cuando se aplica la transformación `One-Hot`, se genera un `numpy array`. Esto pasa con otros transformadores también. Tenemos entonces que transformar nuestros datos de nuevo en un dataframe.

In [ ]:
feature_names = pipeline_regresion.named_steps["preprocesamiento"].get_feature_names_out()
Xt_train_df = pd.DataFrame(
    Xt_train.toarray() if hasattr(Xt_train, "toarray") else Xt_train,
    columns=feature_names,
    index=X_train.index
)

## 6. Entrenamiento del modelo de regresión lineal.

Ahora, pasamos a entrenar el modelo con el conjunto de entrenamiento. Usamos la función `.fit()` para entrenar y aprender los parámetros con base en los datos. Primero, creamos un objeto de la clase `LinearReEgression`.

In [ ]:
Modelo = LinearRegression()

In [ ]:
Modelo.fit(Xt_train_df,y_train)

A continuación, se solicita al modelo que genere las predicciones sobre el conjunto de entrenamiento.

In [ ]:
y_train_pred = Modelo.predict(Xt_train_df)

### Validación del modelo

Para evaluar el desempeño del modelo, analizamos qué tan alejadas están las predicciones respecto a los valores reales de la variable objetivo.

**Mean Absolute Error (MAE)**

MAE mide el error promedio de las predicciones como el valor absoluto de la diferencia entre los valores reales y los estimados. Se interpreta en las mismas unidades de la variable objetivo y no penaliza de forma desproporcionada los errores grandes.

In [ ]:
mae_train = mean_absolute_error(y_train, y_train_pred)
print("MAE  train:", mae_train)

**Root Mean Squeared Error (RMSE)**

El RMSE refleja el error promedio de las predicciones en las mismas unidades de la variable objetivo. Al derivarse del MSE, hereda su mayor penalización a los errores grandes, pero ofrece una interpretación más directa de la magnitud del error.

In [ ]:
mse_train = mean_squared_error(y_train, y_train_pred)
rmse_train = np.sqrt(mse_train)
print("RMSE train:", rmse_train)

**Coeficiente de  determinación R^2**

El coeficiente de determinación (R²) indica qué proporción de la variabilidad de la variable objetivo es explicada por el modelo. En este caso, expresa el porcentaje de la variación en los precios que puede atribuirse a las variables predictoras. Su valor suele estar entre 0 y 1, donde valores cercanos a 1 indican un mejor ajuste del modelo. Un valor cercano a 0 sugiere que el modelo explica muy poca variabilidad, mientras que valores negativos indican que el modelo tiene un desempeño peor que un modelo trivial que predice siempre el valor medio de la variable objetivo.

In [ ]:
r2_train = r2_score(y_train, y_train_pred)
print("R²   train:", r2_train)

## 7. Estimación de la capacidad de generalización del modelo.

A continuación, evaluamos el modelo utilizando el conjunto de prueba (**test**), lo que permite analizar su capacidad de generalización al enfrentarse a datos no vistos durante el entrenamiento. Primero, se utiliza `pipeline_regresion.transform(X_test)`, que aplica automáticamente las transformaciones aprendidas en la fase de entrenamiento. En esta etapa se emplea exclusivamente el método `.transform()`, ya que no se vuelven a ajustar el paso de preparación de datos al test.


In [ ]:
Xt_test = pipeline_regresion.transform(X_test)

In [ ]:
feature_names = pipeline_regresion.named_steps["preprocesamiento"].get_feature_names_out()
Xt_test_df = pd.DataFrame(
    Xt_test.toarray() if hasattr(Xt_test, "toarray") else Xt_test,
    columns=feature_names,
    index=X_test.index
)

Ahora, utilizamos el modelo para predecir sobre el conjunto de prueba:

In [ ]:
y_test_pred = Modelo.predict(Xt_test_df)

### Validación del modelo

Ahora, veamos como se comporta el modelo ante nuevos datos:

In [ ]:
mae_test = mean_absolute_error(y_test, y_test_pred)
print("MAE  test :", mae_test)

In [ ]:
mse_test = mean_squared_error(y_test, y_test_pred)
rmse_test = np.sqrt(mse_test)
print("RMSE test :", rmse_test)

In [ ]:
r2_test = r2_score(y_test, y_test_pred)
print("R²   test :", r2_test)


En el conjunto de prueba, el modelo de regresión lineal obtiene un MAE de 140.596, un RMSE de 225.292 y un R² de 0.66 , lo que indica que, en promedio, los errores en la estimación del precio se sitúan alrededor de 140 mil unidades monetarias, mientras que los errores típicos, más sensibles a desviaciones grandes, rondan los 225 mil. El R² de 0.66 muestra que el modelo explica aproximadamente el 66 % de la variabilidad del precio en datos no vistos durante el entrenamiento. Al comparar estos resultados con los obtenidos en el conjunto de entrenamiento (MAE de 140.066, RMSE de 210.573 y R² de 0.65), se observa que las métricas son similares, e incluso el R² es ligeramente mayor en test, lo que sugiere que el modelo generaliza bien.


## 8. Verificación de supuestos

El componente de supuestos nos ayuda principalmente a interpretar los coeficientes. Recordemos que un modelo de regresión lineal tiene esta forma:

$$ y = {\beta_{0} + \beta_{1}x_{1} + \beta_{2}x_{2} + \beta_{3}x_{3} + \beta_{4}x_{4} + \beta_{5}x_{5}} $$

siendo $\beta_{0}$ el intercepto (bias) y $\beta_{1}$, $\beta_{2}$, $\beta_{3}$, $\beta_{4}$ y $\beta_{5}$ los coeficientes o parámetros correspondientes a las variables de entrada en el mismo orden.

Estos coeficientes nos dicen que tanto impactan las variables explicativas en la estimación del `Precio`. Para ello veamos los coeficientes que se obtuvieron del modelo:


In [ ]:
print("Intercepto:", Modelo.intercept_)

In [ ]:
feature_names = pipeline_regresion.named_steps["preprocesamiento"].get_feature_names_out()
feature_names = [
    name.replace("num__", "").replace("cat__", "")
    for name in feature_names
]
coef_df = pd.DataFrame({
    "Variable": feature_names,
    "Coeficiente": Modelo.coef_
})
coef_df

### Multicolinealidad
La multicolinealidad ocurre cuando dos o más variables independientes están altamente correlacionadas. Aunque no constituye un supuesto fundamental de la regresión lineal, sí es una condición importante para la interpretación confiable de los coeficientes.

Dado que la multicolinealidad afecta principalmente la estabilidad e interpretación de los coeficientes, es necesario contar con un indicador que permita medirla de forma cuantitativa. Para ello se suele utiliza el Factor de Inflación de la Varianza (VIF, por sus siglas en inglés), el cual mide cuánto se incrementa la varianza de un coeficiente de regresión debido a la correlación con las demás variables independientes. Valores de VIF cercanos a 1 indican ausencia de multicolinealidad, mientras que valores altos sugieren una fuerte relación entre las variables y, por tanto, coeficientes menos confiables. De manera general, un VIF mayor o igual a 5 se considera una señal de multicolinealidad significativa.

$$
VIF = \frac{1}{1 - R^2}
$$

Tendremos un valor de **VIF** por cada una de las variables predictoras que, por ejemplo, se puede interpretar de la siguiente forma: 

* $VIF > 4$: Se tiene problema de Multicolinealidad

* $VIF <= 4$: No hay problema de Multicolinealidad

Para nuestro caso se tiene:

In [ ]:
X_vif = Xt_train_df.select_dtypes(include="number").copy()
clean_columns = (
    X_vif.columns
         .str.replace("^num__", "", regex=True)
         .str.replace("^cat__", "", regex=True)
)
vif_values = []
with np.errstate(divide="ignore", invalid="ignore"):
    for i in range(X_vif.shape[1]):
        vif = variance_inflation_factor(X_vif.values, i)
        vif_values.append(vif)

vif_data = pd.DataFrame({
    "Variable": clean_columns,
    "VIF": vif_values
})
print(vif_data)

### Normalidad de los errores

Esto supuesto nos dice que los errores del modelo, es decir la diferencia entre el valor real y el predicho, deberían seguir aproximadamente una distribución normal. Esto quiere decir que la mayoría de los errores deben ser pequeños y los errores muy grandes deben ser poco frecuentes y repartidos de forma simétrica. Cumplir con este supuesto nos permite hacer distintas pruebas sobre el modelo y sus coeficientes.

Existen distintas formas de verificar si el supuesto se cumple o no, uno de ellos es usar una prueba de normalidad, como la de Shapiro–Wilk.


In [ ]:
# Residuales en train
residuales = np.array(y_train - y_train_pred)

estat, p_shapiro = shapiro(residuales)
print("Shapiro-Wilk p-value:", p_shapiro)

In [ ]:
# Histograma
plt.figure(figsize=(6, 4))
sns.histplot(residuales, kde=True)
plt.title("Histograma de residuales")
plt.xlabel("Residual")
plt.ylabel("Frecuencia")
plt.show()

# 2. QQ-plot
plt.figure(figsize=(6, 4))
stats.probplot(residuales, dist="norm", plot=plt)
plt.title("QQ-plot de residuales")
plt.show()

### Homecedasticidad

Este supuesto establece que los errores del modelo deben presentar una variabilidad aproximadamente constante para todos los niveles de las variables explicativas, es decir, que el modelo se equivoque de manera similar tanto en propiedades de bajo como de alto precio. Cuando este supuesto no se cumple, se presenta heterocedasticidad, lo que implica que los errores estándar de los coeficientes pueden estar mal estimados y, en consecuencia, las pruebas estadísticas `t` y `F` utilizadas para evaluar la significancia de las variables dejan de ser confiables. Una forma clásica de evaluar este supuesto es mediante la prueba de Breusch–Pagan.


In [ ]:
X_num = Xt_train_df.select_dtypes(include="number").copy()
mask = np.isfinite(X_num).all(axis=1)
X_num_clean = X_num[mask]
resid_clean = residuales[mask]
X_num_const = sm.add_constant(X_num_clean)
bp_stat, bp_pvalue, _, _ = het_breuschpagan(resid_clean, X_num_const)

print("Estadístico Breusch-Pagan:", bp_stat)
print("p-value Breusch-Pagan   :", bp_pvalue)

### Independencia de los errores

El supuesto de independencia dice que los errores del modelo no deberían estar conectados entre sí, basicamente que el error que comete el modelo en una observación no debería influir en el error de la siguiente.
En caso de que el supuesto no se cumpla, se tiene autocorrelación, que dice que los errores se mueven en grupo, haciendo que cualquier prueba estadística sea poco confiable.

Una forma de revisar el cumplimiento del supuesto es usando el estadístico de Durbin–Watson, que se comporta de esta forma:

$$0≤DW≤4$$

+ Si es $0$ o cercano, nos dice que hay problemas de autocorrelación positiva.
+ Si es $2$ significa que no hay autocorrelación (no hay problemas).
+ Si es $4$ o cercano, indica que hay problemas de autocorrelación negativa.
  

In [ ]:
dw = durbin_watson(residuales)
print("Durbin-Watson:", dw)


El estadístico de Durbin–Watson es $≈ 2.02$, un valor muy cercano a $2$. Esto indica que no se detecta autocorrelación entre los errores.Esto nos dice que los errores se comportan como ruido más o menos independiente entre observaciones. Por lo tanto, con este resultado podemos considerar que el supuesto de independencia de los errores sí se cumple en el modelo.


### Gráficas para ver de manera visual la homocedasticidad e independencia de los residuos

In [ ]:
# Gráficas para ver de manera visual la homocedasticidad e independencia de los residuos

nombres_x = " + ".join(Xt_train_df.columns)

df_graficas=Xt_train_df.join(y_train)

# 2. Fit the linear regression model
model = ols(f'Precio ~ {nombres_x}', data=df_graficas).fit()
fitted_values = model.fittedvalues
residuals = model.resid

# Generación de las gráficas
plt.figure(figsize=(14, 5))

# Gráfica de Homocedasticidad
plt.subplot(1, 2, 1)
plt.scatter(fitted_values, residuals, alpha=0.7, color='steelblue', edgecolors='white')
plt.axhline(y=0, color='red', linestyle='--')
plt.title('Residuos vs Valores Ajustados\n(Evaluación de Homocedasticidad)')
plt.xlabel('Valores Ajustados ($\hat{y}$)')
plt.ylabel('Residuos ($e$)')
plt.grid(True, linestyle=':', alpha=0.6)

# Gráfica de Independencia
plt.subplot(1, 2, 2)
plt.plot(range(len(residuals)), residuals, marker='o', linestyle='-', markersize=4, alpha=0.7, color='darkorange')
plt.axhline(y=0, color='red', linestyle='--')
plt.title('Residuos vs Índice de Observación\n(Evaluación de Independencia)')
plt.xlabel('Orden de la observación')
plt.ylabel('Residuos ($e$)')
plt.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()
# plt.savefig('evaluacion_regresion.png')

# Scale-Location plot (optional but very useful for homoscedasticity)
plt.figure(figsize=(6, 5))
standardized_residuals = model.get_influence().resid_studentized_internal
plt.scatter(fitted_values, np.sqrt(np.abs(standardized_residuals)), alpha=0.7, color='seagreen')
plt.title('Gráfico de Scale-Location\n(Homocedasticidad con residuos estandarizados)')
plt.xlabel('Valores Ajustados')
plt.ylabel('$\sqrt{|Residuos Estandarizados|}$')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
# plt.savefig('scale_location.png')

### Linealidad

El supuesto de linealidad dice que la relación entre las variables explicativas y la media de `Precio` se puede describir razonablemente mediante una combinación lineal, es decir que cada variable aporta un efecto proporcional y más o menos constante sobre el precio cuando aumenta en una unidad, manteniendo las demás fijas.

Una forma de evaluar este supuesto es usar una prueba estadística como el test de Rainbow, que devuelve un estadístico y un p-value para comprobar si el modelo lineal se ajusta bien o si hay indicios de que la relación no es lineal.


In [ ]:
X_sm = sm.add_constant(Xt_train_df)
modelo_sm = sm.OLS(y_train, X_sm).fit()

rainbow_stat, rainbow_pvalue = linear_rainbow(modelo_sm)

print("Estadístico Rainbow:", rainbow_stat)
print("p-value Rainbow    :", rainbow_pvalue)


Tenemos que el test de Rainbow dio un estadístico $≈ 1.03$ y un $p-value$ $≈ 0.12$. El p-value es mayor que el umbral de $0.05$, lo que significa que no hay evidencia estadística para rechazar la linealidad, la prueba considera razonable que la relación entre las variables explicativas y el precio pueda describirse mediante un modelo lineal. Por lo que según esa información el supuesto se cumple.
